In [1]:
# !pip install ~/fwiViz/utility_functions/
# !pip install ../utility_functions/

ERROR: Invalid requirement: '/home/jovyan/fwiViz/utility_functions/': Expected package name at the start of dependency specifier
    /home/jovyan/fwiViz/utility_functions/
    ^
Hint: It looks like a path. File '/home/jovyan/fwiViz/utility_functions/' does not exist.


In [1]:
import fwiVis.fwiVis as fv
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain
from bs4 import BeautifulSoup # I mamba installed bs4
import requests

from datetime import date


# sys.path.insert(0, '~/fwiViz/utility_functions/')
# import fwiVis.fwiVis as fv

In [2]:
## Fix weird formatting where the zeros were taken off the datetimes



In [3]:
### NCCS retrival helper functions for getting gridded data

def listFD(url, ext=''):
    page = requests.get(url).text
    #print(page)
    soup = BeautifulSoup(page, 'html.parser')
    return [url + '/' + node.get('href') for node in soup.find_all('a') if node.get('href').endswith(ext)]

## https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.247.biggestFires/
# https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.216.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/
def get_nccs_url(pattern, url = 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.247.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/', ext = 'csv'):    

#url = 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.216.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/'
#ext = 'csv'
    file_list = []
    for file in listFD(url, ext):
        file_list.append(file)

    try_pd = pd.DataFrame(file_list, columns= ["urls"])
    size = try_pd[try_pd.urls.str.contains(pattern)].urls.values.size
    if(size == 0):
        print("No matches found to pattern. Returning None.")
        return(None)
    if(size >= 2):
        print("Multiple matches found:")
        print(try_pd[try_pd.urls.str.contains(pattern)].urls.values)
        raise ValueError()
    url = try_pd[try_pd.urls.str.contains(pattern)].urls.values[0]
    return(url)

In [4]:


### Function fire_timeline
#from datetime import datetime
#dateparse = lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S')

#df = pd.read_csv(infile, )


## Get gridded fwi
# inputs: fireID
#fireID = 8495

def get_gridded_fwi(fireID):
    
    # Get the URL for the file
    fireID = str(fireID)
    pattern = "FWI." + fireID
    url = get_nccs_url(pattern = pattern)
    
    if(url is not None):
        
        # Get the DF
        grid_FWI = pd.read_csv(url)
        # Change names
        grid_FWI = grid_FWI.rename(columns={'INITDATE': 't', 
                                 "0":"FWI",
                                 "1":"FWI_lead_1",
                                 "2":"FWI_lead_2",
                                 "3":"FWI_lead_3",
                                 "4":"FWI_lead_4",
                                 "5":"FWI_lead_5",
                                 "6":"FWI_lead_6",
                                 "7":"FWI_lead_7",
                                 "8":"FWI_lead_8"
                                })
        # Change dates
        grid_FWI.t = grid_FWI.t.astype("datetime64[ns]").dt.strftime('%Y-%m-%d 12:00:00')

        # return
        return(grid_FWI)
    else:
        return(None)

def concat_subsets(files):
    df = []
    for f in files:
        #manyfr = pd.read_csv(f, parse_dates=['t'], date_parser=dateparse)
        manyfr = pd.read_csv(f)
        manyfr = gpd.GeoDataFrame(manyfr)
        
        
        manyfr.t[~manyfr.t.str.contains("12:00:00")] = manyfr.t[~manyfr.t.str.contains("12:00:00")] + " 00:00:00"
        #print(manyfr.t[~manyfr.t.str.contains("12:00:00")])
        
        manyfr.t = manyfr.t.astype("datetime64[ns]")
        #manyfr.t = manyfr.t.astype("str")

        
        #manyfr.t.tz("UCT")
        df.append(manyfr)
    df = pd.concat(df)
    #df.t = df.t.astype("datetime64[ns]")
    return(df)

def concat_subsets_lower_mem(files):
    df = []
    for f in files:
        #manyfr = pd.read_csv(f, parse_dates=['t'], date_parser=dateparse)
        manyfr = pd.read_csv(f)
        manyfr = gpd.GeoDataFrame(manyfr)
        
        
        manyfr.t = manyfr.t.astype("datetime64[ns]")
        #manyfr.t = manyfr.t.astype("str")

        
        #manyfr.t.tz("UCT")
        df.append(manyfr)
    df = pd.concat(df)
    #df.t = df.t.astype("datetime64[ns]")
    return(df)

def get_lt(lt_string = "Lt_CA_Boreal_"):
    files = glob.glob("/projects/old_shared/fire_weather_vis/Lightning_analysis/computed_data/" + lt_string +"*.csv")
    return(concat_subsets_lower_mem(files))


def fire_timeline_get(fireID, 
                  lt,
                  year = '2023',
                  path_region="BOREAL_NRT_3571_DPS" , 
                  check_last = False, 
                  FWI_source = "station", min_days = 10, 
                  FWI_subset_t = True):
    
    '''
    '''
    
    ## Read in the largefire file of the fireID
    try:
        fr = fv.load_large_fire(fireID, year = year, path_region= path_region, s3_path = True) ## Cluster of 2 fires. 
    except Exception as e:
        print("Fire ID cannot be opened:",fireID)
        print(e)
        return(None)
        ## TO DO Filter? 
            ## VIIRS Static source filter?
            ## WUI filter? 
      
    fr = fr.to_crs("3571")
    
    oldest_perim = fr[fr.t == fr.t.max()]
    first_perim = fr[fr.t == fr.t.min()]
    
    if(check_last == True):
        oldest_perim.t = oldest_perim.t.astype("str")
        m = oldest_perim.explore()
        return(m)
        
    
    
    
    #  ## Subset lightning by time and space
    
    # ## TO DO: Figure out which CA ecoregion/province the fire is in and subset lighting by that? 
    # #print("Not yet subseting spatially beyond quebec. Assuming quebec bounding box")
    
    # min_threshold = fr.t.astype('datetime64[ns]').min() - timedelta(days = min_days)
    # possible_lt = lt[lt.t <= fr.t.min()]
    # possible_lt = possible_lt[possible_lt.t >= min_threshold]

    # first_perim.geometry = first_perim.buffer(750*2) ## Two viirs pixels???
    # join_lt = gpd.sjoin(possible_lt, first_perim, predicate = 'within', how = "inner")
    # join_lt["no_strikes_in_time"] = False
    
    # if (len(possible_lt == 0)):
    #     join_lt["no_strikes_in_time"] = True
    
    # if (len(join_lt[join_lt.InterCloud.isna()].InterCloud) == len(join_lt.InterCloud)):
    #     join_lt["num_candidates"] = 0
    #     join_lt["num_strikes"] = len(possible_lt)
    #     join_lt["num_strikes_10_days"] = len(possible_lt)
    # else:
    #     ## Extract "denominator" or the # of strikes from same period
    #     denominator = possible_lt[possible_lt.t >= join_lt.t_left.min()]
    #     denominator = denominator[denominator.t <= join_lt.t_left.max()]
    #     join_lt["num_candidates"] = len(join_lt)
    #     join_lt["num_strikes"] = len(denominator)
    #     join_lt["num_strikes_10_days"] = len(possible_lt)
        
    # ## Get distance to individuals ignitions
    # # fr["perim_rank"] = fr.t.rank()
    # # first_geom = fr[fr.perim_rank == 1].geometry
    # # first_geom = first_geom.iloc[0]
    # # num_starts = len(first_geom.geoms)
    # # for i in range(0, num_starts):
    # #     join_lt["dist_start_" + str(i)] = join_lt.distance(first_geom.geoms[i].centroid)
    # #     print(fr[fr.perim_rank == 1].to_crs("4326").geometry.iloc[0].geoms[i].centroid)
        
    # ## Rank candidate by distance
    # # range_geoms = list(range(0, num_starts))
    # # string = "dist_start_"
    # # columns_dists = [string + str(x) for x in range_geoms]
    # # top = len(join_lt) * 1 # Top 100%. Could cut to smaller range
    # # dist_bool = join_lt[columns_dists].rank() <= top ## NEED a max distance cutoff. 
    # # join_lt["candidate"] = dist_bool.any(axis = 1)
    
    # ## Get raw VIIRS pixel timing
    # date_string = fr.t.astype("datetime64[ns]").max().strftime("%Y%m%d%p")
    # print(date_string)
    # raw_obs_times = fv.raw_pixel_times(int(fireID), date_string = date_string, path_region = path_region)
    # raw_obs_times = raw_obs_times.reset_index()
    
    # ## get station data
    # if(FWI_source == "station"):
    #     print("Assuming Single Quebec Station. 718270-99999.")
    #     st = pd.read_csv("s3://veda-data-store-staging/EIS/other/station-FWI/19900101.NRT/FWI/718270-99999.linear.HourlyFWIFromHourlyInterpContinuous.csv") ## Corrected record from Robert
    #     st.HH = st.HH.astype("int")
    #     st.YYYY = st.YYYY.astype("int")
    #     st.MM = st.MM.astype("int")
    #     st.DD = st.DD.astype("int")
    #     st = fv.date_convert(st)
        
    #     st_rm = st[["time", "TEMP_C", 'RH_PERC', 'VPD_HPA', 'WDSPD_KPH',
    #    'PREC_MM', 'SNOWD_M', 'VIS_KM', 'FFMC', 'DMC', 'DC', 'BUI', 'ISI',
    #    'FWI', 'OBSMINUTEDIFF_TEMP', 'OBSMINUTEDIFF_RH', 'OBSMINUTEDIFF_WDSPD',
    #    'ISPRECREPORTED', 'OBSMINUTEDIFF_SNOW', 'OBSMINUTEDIFF_VIS']]
    #     st_rm = st_rm.rename(columns = {"time":"t"})

    elif(FWI_source == "gridded"):
        st_rm = get_gridded_fwi(fireID)
        st_rm.t = st_rm.t.astype("datetime64[ns]")
        
    else:
        #print("No other FWI extraction method ready. Sorry. ")
        raise Exception("No other FWI extraction method ready. Sorry. ")
        
    #### Subset station data by time. 
    if(FWI_subset_t):
        st_rm = st_rm[st_rm.t >= min_threshold]
        st_rm = st_rm[st_rm.t <= fr.t.max()]
    
    ## Do merging of all dfs 
    foo = join_lt[["InterCloud", "t_left", "lat_left", "lon_left", "current_mag", "error_elps", "num_station", "num_candidates", "num_strikes", "num_strikes_10_days", "no_strikes_in_time"]]
    foo = foo.rename(columns = {"t_left":"t", "lat_left":"lat", "lon_left":"lon"})
    foo.t = foo.t.astype('datetime64[ns]')
    raw_obs_times = raw_obs_times.rename(columns={"count": "viirs_pix_count"}) 
    raw_obs_times.t = raw_obs_times.t.astype("datetime64[ns]")
    merged = foo.merge(raw_obs_times, on = ["t"], how = "outer")
        
    fr_rm = fr.rename(columns = {"lat":"lat_centroid", "lon":"lon_centroid"})
    fr_rm.t = fr_rm.t.astype("datetime64[ns]")
    merged = merged.merge(fr_rm, on = ["t"], how = "outer")
    
    merged = merged.merge(st_rm, on = ["t"], how = "outer")
    merged["fireID"] = fireID
    
    ## Find temporal thresholds
    
    first_ig = merged[~merged.InterCloud.isna()].t.min()
    last_ig = merged[~merged.InterCloud.isna()].t.max()
    first_detection = merged[~merged.viirs_pix_count.isna()].t.min()

    
    merged["pre_fire"] = ((merged.t >= last_ig) & (merged.t <=  first_detection)) #### first_ig better????????????
    
    ### TODO:  Growht threshold
    return(merged)
    

def lf_ids(year = None, regnm = 'BOREAL_NRT_3571_DPS'):
    
    diroutdata = "s3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-s3-conus/"

    if year == None:
        year = date.today().year

    if diroutdata.startswith("s3://"):
        # Can't use glob for S3. Use s3.ls instead.
        s3 = s3fs.S3FileSystem(anon=False)
        s3path = os.path.join(diroutdata, regnm, str(year), "Largefire")
        fnms = [f for f in s3.ls(s3path)]


    fnms.sort()
    ids = []
    for f in fnms:
        fnm_lts = os.path.basename(f) 
        one_id = fnm_lts[1:-11]
        ids.append(one_id)
    tmp_ids = pd.DataFrame(ids, columns=["ids"])
    tmp_ids = tmp_ids.ids.unique()
    return(tmp_ids)

def unique(list1):
 
    # insert the list to the set
    list_set = set(list1)
    # convert the set to the list
    unique_list = (list(list_set))
    return(unique_list)

def get_listed_ids(quebec_stats):
    newlist = [x.strip('][\n').split(' ') for x in quebec_stats.fireID.unique()]
    newlist = list(chain(*newlist))
    newlist = [x.replace('\n', ' ') for x in newlist]
    newlist = unique(newlist)
    return(newlist)

# def lf_ids(regnm = 'BOREAL_NRT_3571_DPS', year = None):
    
#     diroutdata = "s3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-s3-conus/"

#     if year == None:
#         year = date.today().year

#     if diroutdata.startswith("s3://"):
#         # Can't use glob for S3. Use s3.ls instead.
#         import s3fs
#         s3 = s3fs.S3FileSystem(anon=False)
#         s3path = os.path.join(diroutdata, regnm, str(year), "Largefire")
#         fnms = [f for f in s3.ls(s3path)]
#     else:
#         fnms = glob(os.path.join(diroutdata, regnm, str(year), "Largefire"))

#     if len(fnms) > 0:
#         #print("yeah")
#         fnms.sort()
#         ids = []
#         for f in fnms:
#             fnm_lts = os.path.basename(f) ## Can't work, no ordering
#             ids.append(fnm_lts[1:-11])

#     ids = unique(ids)
#     return(ids)

In [5]:
# diroutdata = "s3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-s3-conus/"

# if year == None:
#     year = date.today().year

# if diroutdata.startswith("s3://"):
#     # Can't use glob for S3. Use s3.ls instead.
#     s3 = s3fs.S3FileSystem(anon=False)
#     s3path = os.path.join(diroutdata, regnm, str(year), "Largefire")
#     fnms = [f for f in s3.ls(s3path)]


# fnms.sort()
# ids = []
# for f in fnms:
#     fnm_lts = os.path.basename(f) 
#     one_id = fnm_lts[1:-11]
#     ids.append(one_id)
# tmp_ids = pd.DataFrame(ids, columns=["ids"])
# tmp_ids = tmp_ids.ids.unique()
   

In [6]:
station_lat_lon = pd.read_csv("s3://veda-data-store-staging/EIS/other/station-FWI/19900101.NRT/isd-history.csv")


station_lat_lon[station_lat_lon.USAF == "718270"]

,USAF,WBAN,STATION NAME,CTRY,STATE,ICAO,LAT,LON,ELEV(M),BEGIN,END
16715,718270,99999,LA GRANDE RIVIERE,CA,NaN,CYGL,53.625,-77.704,194.8,19770701,20231106


In [7]:
lt = get_lt() 

/tmp/ipykernel_502/1194008603.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(df)


In [8]:
lt = gpd.GeoDataFrame(lt, geometry=gpd.points_from_xy(lt.lon, lt.lat), crs=4326) #4674
prov = fv.ca_prov()
lt = lt.sjoin(prov)
lt = lt.to_crs("3571")

In [9]:
#any(lt.columns.isin(["t"]))

In [10]:
#prov

In [11]:
## SUBSET lt to quebec
lt = lt[lt.prov_name_en == 'Quebec']

In [12]:
## SUBSET lt to quebec
#lt = lt[lt.prov_name_en == 'Quebec']

In [13]:
lt = lt[['InterCloud', 't', 'lat', 'lon', 'current_mag',
       'multiplicity_0', 'accr', 'error_elps', 'num_station', 'geometry']]

In [14]:
#tmp = fire_timeline('615', lt = lt, path_region="QuebecGlobalNRT_DPS") #QuebecGlobalNRT_3571

In [15]:
# date_range = pd.date_range(start = "2023-05-01 12:00:00", end = "2023-07-01 12:00:00", freq="12H")
# #date_range_format = datetime.strptime(date_rage, 
# date_snap = date_range.strftime("%Y%m%d%p")

In [16]:
## Get IDs. These IDs come from csvs made by old_shared/fire_weather_vis/Lightning_analysis/snap_prov_lightning.ipynb
# by going through the snapshot files, doing a spatial join, and collecting IDs. 

files = glob.glob("/projects/old_shared/fire_weather_vis/Lightning_analysis/snap_stats//boreal_snapstats*.csv")


fire_stats = concat_subsets(files)

fire_stats.t.max()

### Subsetting fire stats by largefire record, so don't waste time looking for IDs that we haven't got yet. Wait, not worth it, size a bigger thing anyway.
#fire_stats = fire_stats[fire_stats.t < "2023-07-20 12:00:00"]

#fire_stats = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/snap_stats/boreal_snapstats_20231024.csv")

quebec_stats = fire_stats[fire_stats.prov_name_en == "Quebec"]

tmp_list = get_listed_ids(quebec_stats)

ids_lf = lf_ids( year = "2023")

tmp_list = list(set(tmp_list).intersection(ids_lf))

/tmp/ipykernel_502/1194008603.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  manyfr.t[~manyfr.t.str.contains("12:00:00")] = manyfr.t[~manyfr.t.str.contains("12:00:00")] + " 00:00:00"
/tmp/ipykernel_502/1194008603.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  manyfr.t[~manyfr.t.str.contains("12:00:00")] = manyfr.t[~manyfr.t.str.contains("12:00:00")] + " 00:00:00"
/tmp/ipykernel_502/1194008603.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a

In [17]:
ids_lf

array(['10009', '10013', '10017', ..., '9874', '9926', '9999'],
      dtype=object)

In [18]:
from datetime import date
str(date.today().strftime("%Y%m%d"))

'20240617'

In [19]:
#fire_stats.t[(fire_stats.t > "2023-03-02 00:00:00") & (fire_stats.t < "2023-03-04 00:00:00")]

#fire_stats.t[~fire_stats.t.str.contains("12:00:00")] + " 00:00:00"

In [20]:
#ids = ['12641', '12690','10896','9346']

ids = tmp_list


max_t = "maxT" + str(fire_stats.t.max().strftime("%Y%m%d")) + "_"
min_t = "minT" + str(fire_stats.t.min().strftime("%Y%m%d")) + "_"
print(max_t)
print(min_t)

maxT20230830_
minT20230301_


In [21]:
#get_listed_ids(quebec_stats)

In [22]:
####################### BIG EXTRACTION LOOP ########################

# year = '2023'
# path_region= "BOREAL_NRT_3571_DPS" 
# check_last = False 
# FWI_source = "station" 
# ids = ids[0:5]

# fires = pd.DataFrame()
# for n,i in enumerate(ids, start = 0):
#     try:
#         foo = fire_timeline(i, lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20)
#     except Exception as e:
#         print("Error at ID: ",i)
#         print(e)
#         continue
#     ## Extract the period between 
#     fires = pd.concat([fires, foo])
#     #print(fires)
#         #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#     #fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"20_days_fire_stats_only_718270-99999_" +min_t + max_t + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")

In [23]:
#ids
#ids[ids == '2444']
#ids[ids == "8495"]
print("8495" in ids)
print('2444' in ids)
print("no way" in ids)

True
False
False


In [24]:
pd.DataFrame([4], columns= ["id"])

,id
0,4


In [25]:
#fire_timeline(str(11713), lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20)

In [26]:
# ####################### BIG EXTRACTION LOOP  (1/18/2024) To incorperate gridded data. Doing it for largefire IDS ########################

# year = '2023'
# path_region= "BOREAL_NRT_3571_DPS" 
# check_last = False 
# FWI_source = "gridded" 
# #ids = ids
# #ids = ids[0:5]
# error_ids = pd.DataFrame()

# fires = pd.DataFrame()
# for n,i in enumerate(ids, start = 0):
#     try:
#         foo = fire_timeline(str(i), lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20, FWI_subset_t = False)
#     except Exception as e:
#         print("Error at ID: ",i)
#         bad_id = pd.DataFrame([i], columns= ["id"])
#         error_ids = pd.concat([error_ids, bad_id])
#         error_ids.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/" +"Error_ids_from_"+ "Quebec_only_GRIDDED_full_time_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")
#         print(e)
#         continue
#     ## Extract the period between 
#     fires = pd.concat([fires, foo])
#     #print(fires)
#         #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#     fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/" + "Quebec_only_GRIDDED_full_time_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")
    

In [27]:
# Check what IDS are missing an re-run

#path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/againQuebec_only_GRIDDED_20_days_BOREAL_NRT_3571_DPSgridded20240122.csv" ## looking for lightning 20 days before start
path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_only_GRIDDED_full_time_BOREAL_NRT_3571_DPSgridded20240206.csv"
tmp_fires = fv.prep_fire_files(path)

missed_ids = set(tmp_list).symmetric_difference(tmp_fires.fireID.unique())

In [28]:
error_ids = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Error_ids_from_Quebec_only_GRIDDED_full_time_BOREAL_NRT_3571_DPSgridded20240206.csv")

In [29]:
print(len(tmp_fires.fireID.unique()))

print(len(error_ids))

print(len(missed_ids))

161
86
86


In [30]:
set(missed_ids).symmetric_difference(error_ids.id.astype("str")) # They are the same

set()

In [31]:
error_ids

,Unnamed: 0,id
0,0,9791
1,0,16528
2,0,8570
3,0,10729
4,0,12138
...,...,...
81,0,8677
82,0,8747
83,0,15929
84,0,16644


In [32]:
example = fire_timeline(str(15929), lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20, FWI_subset_t = False)

NameError: name 'year' is not defined

In [ ]:
### Redo-ing extraction with the missing IDs, The few I tried worked without error, so maybe a memory issue? 

# year = '2023'
# path_region= "BOREAL_NRT_3571_DPS" 
# check_last = False 
# FWI_source = "gridded" 
# #ids = error_ids.id
# #ids = ids[0:5]
# error_ids = pd.DataFrame()

# fires = pd.DataFrame()
# for n,i in enumerate(ids, start = 0):
#     try:
#         foo = fire_timeline(str(i), lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20, FWI_subset_t = False)
#     except Exception as e:
#         print("Error at ID: ",i)
#         print(e)
#         bad_id = pd.DataFrame([i, str(e)], columns= ["id", "error"])
#         error_ids = pd.concat([error_ids, bad_id])
#         error_ids.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/" +"Error_ids_from_"+ "Quebec_only_GRIDDED_full_time_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")
#         print(e)
#         continue
#     ## Extract the period between 
#     fires = pd.concat([fires, foo])
#     #print(fires)
#         #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#     fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/" + "Quebec_only_GRIDDED_full_time_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")
    

In [ ]:
path1 = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_only_GRIDDED_full_time_BOREAL_NRT_3571_DPSgridded20240206.csv"
tmp_fires1 = fv.prep_fire_files(path1)

path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_only_GRIDDED_full_time_BOREAL_NRT_3571_DPSgridded20240208.csv"
tmp_fires = fv.prep_fire_files(path)

missed_ids = set(tmp_list).symmetric_difference([*tmp_fires.fireID.unique(), *tmp_fires1.fireID.unique()])

In [ ]:
missed_ids

In [ ]:
# year = '2023'
# path_region= "BOREAL_NRT_3571_DPS" 
# check_last = False 
# FWI_source = "gridded" 

# #ids = ids[0:5]
# error_ids = pd.DataFrame()

# fires = pd.DataFrame()
# for n,i in enumerate(missed_ids, start = 0):
#     try:
#         foo = fire_timeline(str(i), lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20, FWI_subset_t = False)
#     except Exception as e:
#         print("Error at ID: ",i)
#         print(e)
#         bad_id = pd.DataFrame([i, str(e)], columns= ["id", "error"])
#         error_ids = pd.concat([error_ids, bad_id])
#         error_ids.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/" +"Error_ids_from_"+ "Quebec_only_GRIDDED_full_time_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")
#         print(e)
#         continue
#     ## Extract the period between 
#     fires = pd.concat([fires, foo])
#     #print(fires)
#         #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#     fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/" + "Second_Quebec_only_GRIDDED_full_time_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")
    

In [ ]:
path1 = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_only_GRIDDED_full_time_BOREAL_NRT_3571_DPSgridded20240206.csv"
tmp_fires1 = fv.prep_fire_files(path1)

path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_only_GRIDDED_full_time_BOREAL_NRT_3571_DPSgridded20240208.csv"
tmp_fires = fv.prep_fire_files(path)

path2 = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Second_Quebec_only_GRIDDED_full_time_BOREAL_NRT_3571_DPSgridded20240208.csv"
tmp_fires2 = fv.prep_fire_files(path2)

missed_ids2 = set(tmp_list).symmetric_difference([*tmp_fires.fireID.unique(), *tmp_fires1.fireID.unique(), *tmp_fires2.fireID.unique()])

missed_ids2

## None missing

In [ ]:
big_final_dataset = pd.concat([tmp_fires, tmp_fires1, tmp_fires2])
big_final_dataset.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv")

In [ ]:
# year = '2023'
# path_region= "BOREAL_NRT_3571_DPS" 
# check_last = False 
# FWI_source = "gridded" 

# ids = missed_ids

# fires = pd.DataFrame()
# for n,i in enumerate(ids, start = 0):
#     try:
#         foo = fire_timeline(str(i), lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20)
#     except Exception as e:
#         print("Error at ID: ",i)
#         print(e)
#         continue
#     ## Extract the period between 
#     fires = pd.concat([fires, foo])
#     #print(fires)
#         #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#     fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"missed_ids"+"Quebec_only_GRIDDED_20_days_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")

In [ ]:
"/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"Quebec_only_GRIDDED_20_days_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv"

# IDS that firetimeline extraction is skipping
8622 - Is getting skipped. No larefirefiles exist except 18622. Where did the ID come from then? Overwritten?
14663 - Also none exist. 
12153 - fails only at the 'get_gridded_FWI' stage (although error says function is not defined). But not ls using asw ls. 

In [ ]:
lf = fv.load_large_fire('12146', year = year, path_region= path_region, s3_path = True)
#get_gridded_fwi("16062")
lf.t = lf.t.astype("str")
lf.explore()

In [ ]:
missing_gridded_ids = []
other_error_ids = [] # Ids where we don't know why it was skipped during "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/againQuebec_only_GRIDDED_20_days_BOREAL_NRT_3571_DPSgridded20240122.csv" generation

for i in missed_ids:
    tmp = get_gridded_fwi(i)
    if tmp is None:
        missing_gridded_ids.append(i)
    else:
        print(tmp)
        other_error_ids.append(i)

In [ ]:
print(other_error_ids)
missing_gridded_ids

In [ ]:
##Check if the IDS listed below were in the ones that I gave to rboert

lf_centroids_from_nov_8 = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/lf_centroids/Nov_8_fire_centroids_Quebec_from_boreal_snapstatsBOREAL_NRT_3571_DPS(1).csv")

print(lf_centroids_from_nov_8[lf_centroids_from_nov_8.fireID.astype("str").isin(missing_gridded_ids)].fireID.unique())
print(lf_centroids_from_nov_8[lf_centroids_from_nov_8.fireID.astype("str").isin(other_error_ids)].fireID.unique())
print(lf_centroids_from_nov_8[lf_centroids_from_nov_8.fireID.astype("str").isin(["8495"])].fireID.unique())

In [ ]:
# ### Quickly append the one missing ID

# year = '2023'
# path_region= "BOREAL_NRT_3571_DPS" 
# check_last = False 
# FWI_source = "gridded" 

# ids =  other_error_ids

# fires = pd.DataFrame()
# for n,i in enumerate(ids, start = 0):
#     try:
#         foo = fire_timeline(str(i), lt = lt, year = year, path_region= path_region, check_last = False, FWI_source = FWI_source, min_days = 20)
#     except Exception as e:
#         print("Error at ID: ",i)
#         print(e)
#         continue
#     ## Extract the period between 
#     fires = pd.concat([fires, foo])
#     #print(fires)
#         #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#     fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"missed_ids"+"Quebec_only_GRIDDED_20_days_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")

In [ ]:
get_gridded_fwi('8553')

In [ ]:
"/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"20_days_all_lf_ids_" + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv"

In [ ]:
path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/20_days_all_lf_ids_BOREAL_NRT_3571_DPSgridded20240119.csv" ## looking for lightning 20 days before start

from shapely import wkt




#gpd_fires = gpd.GeoDataFrame(fires, crs = "3571", geometry  = 'geometry')


fires = pd.read_csv(path)
fires

In [ ]:
fires = fires.rename(columns={"geometry":"csv_geometry"})
fires.t = fires.t.astype("str")
fires.fireID  = fires.fireID.astype("str")
#fires['csv_geometry'] =fires['csv_geometry'].apply(wkt.loads)
fires_geom = gpd.read_file(path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
fires_geom.t = fires_geom.t.astype("str")
fires_geom.fireID  = fires_geom.fireID.astype("str")

fires = fires_geom[["fireID", "t", "geometry"]].merge(fires, on=["fireID", "t"], how = "left")

#fires = gpd.GeoDataFrame(fires, crs = "3571", geometry = "csv_geometry")
fires.columns

In [ ]:
#path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/fire_stats_only_718270-99999_minT2023-03-01_12:00:00_maxT2023-08-30_12:00:00_BOREAL_NRT_3571_DPSstation20231101.csv" ## Original 'compare to station' csv. 10 days before lightining

path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/20_days_fire_stats_only_718270-99999_minT20230301_maxT20230830_BOREAL_NRT_3571_DPSstation20231120.csv" ## looking for lightning 20 days before start

from shapely import wkt




#gpd_fires = gpd.GeoDataFrame(fires, crs = "3571", geometry  = 'geometry')


fires = pd.read_csv(path)

fires = fires.rename(columns={"geometry":"csv_geometry"})
fires.t = fires.t.astype("str")
fires.fireID  = fires.fireID.astype("str")
#fires['csv_geometry'] =fires['csv_geometry'].apply(wkt.loads)
fires_geom = gpd.read_file(path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO")
fires_geom.t = fires_geom.t.astype("str")
fires_geom.fireID  = fires_geom.fireID.astype("str")

fires = fires_geom[["fireID", "t", "geometry"]].merge(fires, on=["fireID", "t"], how = "left")

#fires = gpd.GeoDataFrame(fires, crs = "3571", geometry = "csv_geometry")
fires.columns

In [ ]:
#fires[~fires.geometry.isna()]

In [ ]:
fires[["fireID",'InterCloud',"t" ,'num_candidates', 'num_strikes', 'num_strikes_10_days',
       'viirs_pix_count', "geometry"]]

In [ ]:
fires.InterCloud.unique()

In [ ]:
#fire_timeline(fireID = '13165' , lt = lt, year = year, path_region= path_region , check_last = False, FWI_source = FWI_source)

In [ ]:
# "20230720PM"

# raw_obs_times = fv.raw_pixel_times(int(12641), date_string = "20230720PM", path_region = "BOREAL_NRT_3571_DPS")

In [ ]:
# # import pickle

# fireID = '12641'
# file = open('/projects/shared-buckets/gsfc_landslides/FEDSoutput-s3-conus/BOREAL_NRT_3571_DPS/2023/Serialization/20230720PM.pkl', 'rb')

# # dump information to that file
# data = pickle.load(file)

# # close the file
# file.close()

# fireID = int(fireID)

# times = []
# for i in range(0, len(data.fires[fireID].pixels)):
#     #print(i)
#     times.append(data.fires[fireID].pixels[i].datetime)

In [ ]:
#len(data.fires)

In [ ]:
#len(data.fires[fireID].pixels)

In [ ]:
# import PIL
# from pathlib import Path
# from PIL import UnidentifiedImageError

# path = Path("INSERT PATH HERE").rglob("*.jpeg")
# for img_p in path:
#     try:
#         img = PIL.Image.open(img_p)
#     except PIL.UnidentifiedImageError:
#             print(img_p)

In [ ]:
# fires = fires.set_crs("3571")
# proj_fires = fires[~fires.geometry.isna()].to_crs("4326")
# proj_fires.t = proj_fires.t.astype("datetime64[ns]")
# proj_fires["days_since"] = (proj_fires.t - proj_fires.t.min())
# proj_fires["days_since"] = proj_fires["days_since"].dt.days

In [ ]:
# print(proj_fires.t.max())
# print(proj_fires.t.min())

In [ ]:
# import matplotlib as mpl

# with mpl.rc_context({'font.size': 30}):
#     plot = proj_fires.plot(facecolor="none",figsize=(20, 20), column = "days_since", 
#                            cmap = 'autumn_r', 
#                            legend=True, 
#                            legend_kwds={'ticks': [], "orientation":"horizontal"},)
#     cx.add_basemap(ax = plot, source=cx.providers.Esri.WorldImagery, attribution = False, crs = proj_fires.crs.to_string())
#                     #crs=gdf.crs.to_string(), source=cx.providers.NASAGIBS.ModisTerraBands367CR)

# plt.savefig('Quebec_largefire.png', dpi = 900, transparent = True)
    

In [ ]:
# ?mpl.rc_context

In [ ]:
#new_fire = fires.groupby("fireID")

In [ ]:
# #group_first_ig = fires[~fires.InterCloud.isna()].groupby("fireID").t.min() #.reset_index()
# group_first_ig

In [ ]:
# group_first_detect = fires[~fires.viirs_pix_count.isna()].groupby("fireID").t.min() #.reset_index()
# group_first_detect

In [ ]:
# smol["pre_fire"] = ((smol.t >= last_ig) & (smol.t <=  first_detection))

In [ ]:
# fires.columns

In [ ]:
# fires.groupby('fireID').t["Inter"]

In [ ]:
#fires.fireID.isin(tmp_list)
#len(fires.fireID.unique())/len(tmp_list)

In [ ]:
#smol = fires[fires.fireID == '12375']

In [ ]:
#subset_group_first_detect = group_first_detect[group_first_detect.index.isin(group_first_ig.index)]

In [ ]:
#fires.groupby('fireID').t >= group_first_ig

In [ ]:
fires.InterCloud.unique

In [ ]:
fires.columns

In [ ]:
met_cols = ['TEMP_C', 'RH_PERC', 'VPD_HPA',
       'WDSPD_KPH', 'PREC_MM', 'SNOWD_M', 'VIS_KM', 'FFMC', 'DMC', 'DC', 'BUI',
       'ISI', 'FWI', 'OBSMINUTEDIFF_TEMP', 'OBSMINUTEDIFF_RH']

In [ ]:
fires.num_strikes.unique()

In [ ]:

IDs_with_strikes = fires[~fires.InterCloud.isnull()].fireID.unique()
nostrike_ids = fires[~fires.fireID.isin(IDs_with_strikes)].fireID.unique()


In [ ]:
print(len(nostrike_ids))
print(len(IDs_with_strikes))

In [ ]:
#percent_no_candidate_ignition = (len(fires.fireID.unique()) - len(fires[~fires.InterCloud.isna()].fireID.unique()))/ len(fires.fireID.unique())
percent_no_candidate_ignition = len(nostrike_ids)/len(fires.fireID.unique())

print("Percent no ignitions: ", percent_no_candidate_ignition)
#print("No ignitions bc of time", len(fires[fires.no_strikes_in_time == True].fireID.unique())) ## Misleading
#print("No ignitions by candidates", len(fires[fires.num_candidates == 0].fireID.unique()))
#print("Candidate ignitions", len(fires[fires.num_candidates.astype("float") > 0].fireID.unique()))
print("Total", len(fires.fireID.unique()))
print("Total possible IDS:" ,len(tmp_list))

group_first_detect = fires[~fires.viirs_pix_count.isna()].groupby("fireID").t.min().reset_index()
group_first_detect = group_first_detect.rename(columns={"t":"first_detect"})
#group_first_detect


group_last_ig = fires[~fires.InterCloud.isna()].groupby("fireID").t.max().reset_index()
group_last_ig = group_last_ig.rename(columns={"t":"last_ig"})

time_dist = group_last_ig.merge(group_first_detect, on = ["fireID"], how = "outer")
time_dist = time_dist.dropna()
time_dist
# group_first_ig

time_dist["time_diff"] = time_dist.first_detect.astype("datetime64[ns]") - time_dist.last_ig.astype("datetime64[ns]")

In [ ]:
import matplotlib as mpl
import matplotlib.dates as mdates

with mpl.rc_context({'font.size': 18}):
    time_dist["time_diff"].dt.days.plot.hist(title = "Days from strike to First VIIRS detection")
plt.savefig('Strike_to_VIIRS.png', dpi = 900, transparent = True,bbox_inches = "tight")
plt.show()
print(len(time_dist))
print(len(tmp_list))
print(len(time_dist)/len(tmp_list))

print(max_t)
print(min_t)
print(min(time_dist.time_diff))
print(max(time_dist.time_diff))
ids_with_big_time_differences = time_dist[time_dist.time_diff.dt.days > 10].fireID
time_dist

In [ ]:
# fr = fr.sort_values(by = ['t'])

# fig, ax = plt.subplots()
# ax.fill_between(upper.index, upper.FWI.rolling(1).mean(), lower.FWI.rolling(5).mean(), 
#                 facecolor='grey', 
#                 alpha=0.2,
#                 label= "95th Percentile")
# ax.fill_between(mid_upper.index, mid_upper.FWI.rolling(1).mean(), mid_lower.FWI.rolling(5).mean(), 
#                 facecolor='grey', 
#                 alpha=0.4,
#                 label= "25th Percentile")
# ax.plot(mean_quant.index, mean_quant.FWI.rolling(1).mean(), 
#         color = "black",
#         label= "Historic Mean Per Day")
# ax.plot(st[(st.time >= "2023-05-01")].time.astype('datetime64[ns]'), st[(st.time >= "2023-05-01")].FWI)
# ax.set_ylabel("Fire Weather Index")
# ax.set_title("2023 Fire Weather Index (FWI) for La Grande Rivière, Quebec, Canada (WMO ID 718270)")
# #ax.legend()
# ax2 = ax.twinx()
# ax2.scatter(candidate.t_left.astype("datetime64[ns]"), candidate.candidate, color = "purple")
# ax2.set_yticklabels("")
# #ax2.plot(fire_stats.t, fire_stats.num_active_fires, color = "red", label = "Number of Fires in Quebec")
# #ax2.legend(loc = 0.5)
# ax3 = ax.twinx()
# #ax3.spines.right.set_position(("axes", 1.2))
# ax3.plot(fr.t.astype("datetime64[ns]"), fr.farea, color = "red")
# #ax3.plot(day_strike.index.astype("datetime64[ns]"), day_strike.InterCloud, color = "orange")

# ax4 = ax.twinx()
# ax4.spines.right.set_position(("axes", 1.2))
# ax4.scatter(raw_obs_times.index, raw_obs_times['count'], color = "orange")

# fig.autofmt_xdate()

# # print("First detection: " + str(fr.t.astype("datetime64[ns]").min()) )
# print(candidate.t_left.min())
# print(candidate.t_left.max())

In [ ]:
# def gpd_read_file(filename, parquet=False, **kwargs):
#     itry = 0
#     maxtries = 5
#     fun = gpd.read_parquet if parquet else gpd.read_file
#     while itry < maxtries:
#         try:
#             dat = fun(filename, **kwargs)
#             return dat
#         except Exception as e:
#             itry += 1
#             print(f"Attempt {itry}/{maxtries} failed.")
#             if not itry < maxtries:
#                 raise e



# def load_large_fire(fireID, year = "2019", path_region = "WesternUS", layer='perimeter', s3_path = False):
#     '''
#     loads in largefire file based on fireID and layer, then preps it for "explore" by adding centriod data. Currently limited to one year. 
    
#     INPUTS:
        
#         fireID (str): fireID offire of interest. Can be found in gdf files read in by prep_gdf and load_file. Can be selected interactivly form a gdf if use gdf.explore()
#         year (str): Year that fires took place. Default to 2019. Availible options differ by path_region. 
#         path_region (str): This constructs the path that the fires are stored in. WesternUS and CONUS availible. 
#         layer (str): The largefire layer to load. Options are 'perimeter', 'nfplist', 'fireline', and 'newfirepix'. 
#     '''
#     if(s3_path == True):
#         tmp = s3.glob('s3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-s3-conus/' + path_region +'/'+ year +'/Largefire/F' + fireID + '_*')
#         lf_files =  ["s3://" + t for t in tmp]
    
#     else:
#         lf_files = glob.glob('/projects/shared-buckets/gsfc_landslides/FEDSoutput-s3-conus/' + path_region +'/'+ year +'/Largefire/F' + fireID + '_*')
#         #print(lf_files)
#     lf_ids = list(set([file.split('Largefire/')[1].split('_')[0] for file in lf_files])) 
#     print(lf_ids)
#     largefire_dict = dict.fromkeys(lf_ids)
    
#     for lf_id in lf_ids:
#         most_recent_file = [file for file in lf_files if lf_id in file][-1]
#         largefire_dict[lf_id] = most_recent_file
#     if(s3_path == True):
#         gdf = pd.concat([gpd_read_file(file,layer= layer) for key, file in largefire_dict.items()], 
#                        ignore_index=True)
#     else:
#         gdf = pd.concat([gpd.read_file(file,layer= layer) for key, file in largefire_dict.items()], 
#                        ignore_index=True)
#     gdf = gdf.to_crs('EPSG:4326')
#     gdf['lon'] = gdf.centroid.x
#     gdf['lat'] = gdf.centroid.y
#     return gdf

In [ ]:
# load_large_fire(fireID = '13165', year = '2023', path_region= "BOREAL_NRT_3571_DPS", s3_path = True)

In [ ]:
# fireID = '13165'

# s3.glob('s3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-s3-conus/' + path_region +'/'+ year +'/Largefire/F' + fireID + '_*')

In [ ]:
# tmp = s3.glob('s3://maap-ops-workspace/shared/gsfc_landslides/FEDSoutput-s3-conus/' + path_region +'/'+ year +'/Largefire/F' + fireID + '_*')

# tmp =  ["s3://" + t for t in tmp]

# tmp

In [ ]:
# fires[["fireID", 't', 'InterCloud', 'num_candidates']][fires.InterCloud.isna()]#.groupby(["fireID", 'InterCloud', 'num_candidates']).t.min()

In [ ]:
fires.columns

In [ ]:
min([np.nan, 0.0])

In [ ]:
fires["InterCloud_new_labs"] = fires.InterCloud
fires.loc[fires.InterCloud.isnull(),"InterCloud_new_labs"] = -1
has_lt = fires.groupby("fireID").InterCloud_new_labs.max().reset_index()
has_lt



In [ ]:


print(len(has_lt[has_lt.InterCloud_new_labs == -1]))

print(len(has_lt[has_lt.InterCloud_new_labs == 0]))


In [ ]:
no_lt_ids = has_lt[has_lt.InterCloud_new_labs == -1].fireID.unique()
no_lt_ids

In [ ]:
# m = fire_timeline('10054', lt = lt, year = year, path_region= path_region, check_last = True, FWI_source = FWI_source)
# m2 = fire_timeline('10056', lt = lt, year = year, path_region= path_region, check_last = True, FWI_source = FWI_source)
# # for i in no_lt_ids:
    
#     m2 = fire_timeline(i, lt = lt, year = year, path_region= path_region, check_last = True, FWI_source = FWI_source)
#     m = m + m2

# m



In [ ]:
fires.columns

In [ ]:

#from shapely import wkt


#fires[~fires.geometry.isna()].geometry = fires[~fires.geometry.isna()].geometry.astype("str").apply(wkt.loads)
gpd_fires = fires #gpd.GeoDataFrame(fires)
gpd_fires
#gpd_fires.set_crs("3571")

In [ ]:
#gpd_fires[gpd_fires.num_candidates > 0]

In [ ]:
gpd_fires.t = gpd_fires.t.astype("str")

In [ ]:
#from shapely import wkt


#fires['geometry'] =fires['geometry'].apply(wkt.loads)

#gpd_fires = gpd.GeoDataFrame(fires, crs = "3571", geometry  = 'geometry')
gpd_fires = gpd_fires.set_crs(crs = "3571")

In [ ]:
#gpd_fires[~gpd_fires.geometry.isna()].explore(column = "InterCloud", style_kwds = {"fillOpacity": "0.1"} )

In [ ]:
#no_null_geoms =

first_perims = gpd_fires[~gpd_fires.geometry.isnull()].groupby("fireID").t.min().reset_index()
first_perims

last_perims = gpd_fires[~gpd_fires.geometry.isnull()].groupby("fireID").t.max().reset_index()
last_perims

print(type(gpd_fires))
mask = gpd_fires.geometry.isna()
trouble = gpd_fires[~(mask)]
#plot_last = gpd_fires[~gpd_fires.geometry.isnull()].merge(last_perims, on = ["fireID", "t"], how = 'right')
#plot_last = last_perims.merge(gpd_fires[~gpd_fires.geometry.isnull()], on = ["fireID", "t"], how = 'left')
#gpd_fires = gpd_fires.set_geometry("geometry")

plot_last = fires.merge(last_perims, on = ["fireID", "t"], how = 'right')
plot_first = fires.merge(first_perims, on = ["fireID", "t"], how = 'right')
#plot_last = fires.merge(first_perims, on = ["fireID", "t"], how = 'right')
plot_last = plot_last.merge(plot_first[["fireID", "t", "geometry", "InterCloud"]], on = ["fireID", "t", "geometry", "InterCloud"], how = 'outer')
plot_last = plot_last.sort_values(by = "fireID", ascending= True)
plot_last[["fireID", "t", "geometry", "InterCloud"]]
plot_last.InterCloud.unique()

In [ ]:
# print(type(trouble))
# print(type(gpd_fires[~gpd_fires.geometry.isnull()]))
# print(type(plot_last))

has_lt = fires.groupby("fireID").InterCloud.max().reset_index()


has_lt["has_lt"] = ~(has_lt.InterCloud.isnull())

In [ ]:
#has_lt[has_lt.has_lt]

In [ ]:
print(has_lt["has_lt"].unique())

plot_last = plot_last.merge(has_lt, on = ["fireID"], how = "left")

In [ ]:
#plot_last = gpd.GeoDataFrame(plot_last)
print(type(plot_last))
plot_last.columns

In [ ]:
# plot_last[['fireID', 't', 'has_lt', 'n_pixels', 'n_newpixels',
#        'farea', 'fperim', 'flinelen', 'duration', 'pixden', 'meanFRP',
#        'geometry', 'lon_centroid', 'lat_centroid', ]].explore(column = "has_lt", cmap = "Set1")

#plot_last[~plot_last.geometry.isnull()].explore(column = "has_lt", cmap = "Set1")

plot_last_tmp = plot_last[~plot_last.geometry.isnull()]
#plot_last_tmp.geometry


#from shapely import wkt


#plot_last_tmp['geometry'] = plot_last_tmp['geometry'].apply(wkt.loads)

plot_last_tmp = gpd.GeoDataFrame(plot_last_tmp, crs = "3571", geometry  = 'geometry')
type(plot_last_tmp)

# print(plot_last_tmp.InterCloud.unique())
# print(len(plot_last_tmp[plot_last_tmp.InterCloud == '']))
# print(len(plot_last_tmp[plot_last_tmp.InterCloud.isnull()]))

In [ ]:
fires[fires.fireID == "8495"]

In [ ]:
### Read in CIFFC data

ciffc = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/CIFFC_data/ciffc_all_canada.csv")
ciffc = ciffc[ciffc.field_agency_code == "qc"]

ciffc = gpd.GeoDataFrame(ciffc, geometry= gpd.points_from_xy(ciffc.field_longitude, ciffc.field_latitude), crs = "4326")
ciffc = ciffc.to_crs("3571")
plot_last_tmp = plot_last_tmp.sjoin(ciffc)

In [ ]:
ciffc.field_agency_code.unique()

In [ ]:
ciffc[ciffc.field_fire_size >= 500].groupby("field_system_fire_cause").count()

In [ ]:
ciffc["is_natural"] = ciffc.field_system_fire_cause == "N"

In [ ]:
plot_first = plot_first.set_crs("3571")

m = plot_last_tmp[plot_last_tmp.fireID.isin(ids_with_big_time_differences)][['t', 'viirs_pix_count', 'fireID', 'n_pixels',
       'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration', 'pixden',
       'meanFRP', 'geometry', 'has_lt']].explore()

m = plot_first[plot_first.fireID.isin(ids_with_big_time_differences)][['t', 'viirs_pix_count', 'fireID', 'n_pixels',
       'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration', 'pixden',
       'meanFRP', 'geometry']].explore(m = m)


# st = station_lat_lon[station_lat_lon.USAF == "718270"]

# add_st_dot = gpd.GeoDataFrame(st, 
#                               geometry= gpd.points_from_xy(st.LON, st.LAT))
# m = add_st_dot.explore(m = m, style_kwds = {"fillOpacity": "0.1"})

m = ciffc[ciffc.field_fire_size >= 500].explore(m = m, column = "is_natural",  cmap = "Dark2")

m

#m.save("ciffc_vs_final_feds_periemter.html")

In [ ]:
m = plot_last_tmp[['t', 'viirs_pix_count', 'fireID', 'n_pixels',
       'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration', 'pixden',
       'meanFRP', 'geometry', 'has_lt']].explore(column = "has_lt", cmap = "Set1")
# st = station_lat_lon[station_lat_lon.USAF == "718270"]

# add_st_dot = gpd.GeoDataFrame(st, 
#                               geometry= gpd.points_from_xy(st.LON, st.LAT))
# m = add_st_dot.explore(m = m, style_kwds = {"fillOpacity": "0.1"})

m = ciffc[ciffc.field_fire_size >= 500].explore(m = m, column = "is_natural",  cmap = "Dark2")

m

In [ ]:
plot_last_tmp["is_natural"] = plot_last_tmp.field_system_fire_cause == "N"
#plot_last_tmp

In [ ]:
m = plot_last_tmp[['t', 'viirs_pix_count', 'fireID', 'n_pixels',
       'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration', 'pixden',
       'meanFRP', 'geometry', 'has_lt', "is_natural"]].explore(column = "is_natural", cmap = "Set3")
# st = station_lat_lon[station_lat_lon.USAF == "718270"]

# add_st_dot = gpd.GeoDataFrame(st, 
#                               geometry= gpd.points_from_xy(st.LON, st.LAT))
# add_st_dot.explore(m = m, style_kwds = {"fillOpacity": "0.1"})
m

In [ ]:
# plot_last_tmp[plot_last_tmp.fireID == '10373'][['t', 'viirs_pix_count', 'fireID', 'n_pixels',
#        'n_newpixels', 'farea', 'fperim', 'flinelen', 'duration', 'pixden',
#        'meanFRP', 'geometry', 'has_lt']].explore(column = "has_lt", cmap = "Set1")

In [ ]:
## Check timing of the lightning vs ciffc reports

group_first_detect = fires[~fires.viirs_pix_count.isna()].groupby("fireID").t.min().reset_index()
group_first_detect = group_first_detect.rename(columns={"t":"first_detect"})
#group_first_detect
group_first_detect2 =  fires[~fires.geometry.isna()].groupby("fireID").t.max().reset_index()

group_last_ig = fires[~fires.InterCloud.isna()].groupby("fireID").t.max().reset_index()
group_last_ig = group_last_ig.rename(columns={"t":"last_ig"})

time_dist = group_last_ig.merge(group_first_detect, on = ["fireID"], how = "outer")
time_dist = time_dist.dropna()


In [ ]:
first_detect_with_geom = fires[~fires.geometry.isna()].merge(group_first_detect2, on = ["fireID", "t"], how = "right")
first_detect_with_geom = first_detect_with_geom[["fireID", "geometry"]]
first_detect_with_geom = group_first_detect.merge(first_detect_with_geom, on = ["fireID"])

first_detect_with_last_geom_no_lt = first_detect_with_geom
first_detect_with_geom =  first_detect_with_geom.merge(time_dist, on = ["fireID", "first_detect"])


first_detect_with_last_geom_no_lt = gpd.GeoDataFrame(first_detect_with_last_geom_no_lt, crs = "3571")
first_detect_with_geom = gpd.GeoDataFrame(first_detect_with_geom, crs = "3571")
ciffc_with_first = first_detect_with_geom.sjoin(ciffc[ciffc.field_fire_size >= 500])
#ciffc_with_first
ciffc_with_first["diff_first_detect_report_date"] = ciffc_with_first.first_detect.astype("datetime64[ns]") - ciffc_with_first.field_situation_report_date.astype("datetime64[ns]")
ciffc_with_first["diff_last_strike_report_date"] = ciffc_with_first.last_ig.astype("datetime64[ns]") - ciffc_with_first.field_situation_report_date.astype("datetime64[ns]")


ciffc_all = first_detect_with_last_geom_no_lt.sjoin(ciffc[ciffc.field_fire_size >= 500])
ciffc_all["diff_first_detect_report_date"] = ciffc_all.first_detect.astype("datetime64[ns]") - ciffc_all.field_situation_report_date.astype("datetime64[ns]")


In [ ]:
ciffc_with_first[~(ciffc_with_first.fireID == "12155")].diff_first_detect_report_date.dt.days.plot.hist(title = "Days from first report to First VIIRS detection", 
                                                                                                       color = "orange")
print(len(ciffc_with_first["diff_first_detect_report_date"]))
print(min(ciffc_with_first["diff_first_detect_report_date"]))
print(max(ciffc_with_first["diff_first_detect_report_date"]))
ciffc_with_first[ciffc_with_first.diff_first_detect_report_date > "8000 days" ] ### Ok, clearly wrong report date. 

In [ ]:
with mpl.rc_context({'font.size': 18}):
    ciffc_all[~(ciffc_all.diff_first_detect_report_date > "6000 days")].diff_first_detect_report_date.dt.days.plot.hist(title = "Days from first report to First VIIRS detection", color = "orange")
    plt.savefig('ciffc_to_VIIRS.png', dpi = 900, transparent = True,bbox_inches = "tight")
    plt.show()

In [ ]:
                                                                                                       color = "orange")
print(len(ciffc_all["diff_first_detect_report_date"]))
print(min(ciffc_all["diff_first_detect_report_date"]))
print(max(ciffc_all["diff_first_detect_report_date"]))
ciffc_all[ciffc_all.diff_first_detect_report_date > "8000 days" ] ### Ok, clearly wrong report date. 

In [ ]:
days_inverse = ciffc_with_first[~(ciffc_with_first.fireID == "12155")].diff_last_strike_report_date * -1
ciffc_with_first["inverse_strike_report_days"] = ciffc_with_first.diff_last_strike_report_date.dt.days * -1

In [ ]:
ciffc_with_first[~(ciffc_with_first.fireID == "12155")].diff_last_strike_report_date.dt.days
#ciffc_with_first.inverse_strike_report_days
#ciffc_with_first.inverse_strike_report_days = ciffc_with_first.inverse_strike_report_days.astype("float64")

In [ ]:
with mpl.rc_context({'font.size': 18}):
    #ciffc_with_first[~(ciffc_with_first.fireID == "12155")].diff_last_strike_report_date.dt.days.plot.hist(title = "Days from first strike to first report", color = "purple")
    ciffc_with_first[(~(ciffc_with_first.fireID == "12155")) & (ciffc_with_first.inverse_strike_report_days >=0)].inverse_strike_report_days.plot.hist(title = "Days from first strike to first report", color = "purple")
    plt.savefig('strike_to_ciffc.png', dpi = 900, transparent = True,bbox_inches = "tight")
    plt.show()                                                                                      

print(min(ciffc_with_first["diff_last_strike_report_date"]))
print(max(ciffc_with_first["diff_last_strike_report_date"]))

In [ ]:
ciffc_with_first.diff_first_detect_report_date = ciffc_with_first.diff_first_detect_report_date.astype("str")
ciffc_with_first.diff_last_strike_report_date = ciffc_with_first.diff_last_strike_report_date.astype("str")
ciffc_with_first.field_situation_report_date = ciffc_with_first.field_situation_report_date.astype("str")
ciffc_with_first.field_status_date = ciffc_with_first.field_status_date.astype("str")
ciffc_with_first.explore()

In [ ]:
time_dist["time_diff"] = time_dist.first_detect.astype("datetime64[ns]") - time_dist.last_ig.astype("datetime64[ns]")

In [ ]:
from datetime import timedelta

### Look at the different distributions of FWI acording to station data. Extract some possible FWIs'

# - FWI from between last_ig and first detect, after first detect
# - FWI from just after last_ig and before first detect as 1/2 

lt_ids = time_dist.fireID.unique()
fires["is_period_between_ig_fd"] = None
fires["is_period_after_fd"] = None
fires["is_day_after_ig_or_smaller"] = None
fires["is_day_after_fd"] = None



for i in lt_ids:
    last_ig_day =  time_dist[time_dist.fireID == i].last_ig.astype("datetime64[ns]") + timedelta(days = 1)
    fd_day =  time_dist[time_dist.fireID == i].first_detect.astype("datetime64[ns]") + timedelta(days = 1)
    tmp = (time_dist[time_dist.fireID == i].first_detect.astype("datetime64[ns]") + time_dist[time_dist.fireID == i].time_diff)
    fires.loc[fires.fireID == i, "is_period_between_ig_fd"] = (fires[fires.fireID == i].t >= str(*time_dist[time_dist.fireID == i].last_ig.values)) & (fires[fires.fireID == i].t < str(time_dist[time_dist.fireID == i].first_detect.values))
    fires.loc[fires.fireID == i, "is_period_after_fd"] = (fires[fires.fireID == i].t >= str(*time_dist[time_dist.fireID == i].first_detect.values)) & (fires[fires.fireID == i].t <= str(*tmp.values))
    fires.loc[fires.fireID == i, "is_day_after_ig_or_smaller"] = (fires[fires.fireID == i].t >= str(*time_dist[time_dist.fireID == i].last_ig.values)) & ((fires[fires.fireID == i].t < str(time_dist[time_dist.fireID == i].first_detect.values)) & (fires[fires.fireID == i].t <= str(*last_ig_day.values)))
    fires.loc[fires.fireID == i, "is_day_after_fd"] = (fires[fires.fireID == i].t >= str(*time_dist[time_dist.fireID == i].first_detect.values)) & (fires[fires.fireID == i].t <= str(*fd_day.values))

In [ ]:
 # time_dist[time_dist.fireID == i].last_ig.astype("datetime64[ns]") + timedelta(days = 1)

In [ ]:
#fires[fires.fireID == "8563"][["fireID", "t","is_period_between_ig_fd", "is_period_after_fd"]]

In [ ]:
#lt_ids

In [ ]:
#str(time_dist[time_dist.fireID == "10140"].first_detect.values)

In [ ]:
#fires.loc[fires.fireID == "10140", "is_period_after_fd"] = 
#(fires[fires.fireID == "10140"].t >= str(time_dist[time_dist.fireID == "10140"].first_detect.values))# & (fires[fires.fireID == i].t <= str(*tmp.values))

In [ ]:
# i = "13150"

# tmp = (time_dist[time_dist.fireID == i].first_detect.astype("datetime64[ns]") + time_dist[time_dist.fireID == i].time_diff)

# tmp_early = (fires[fires.fireID == i].t >= str(*time_dist[time_dist.fireID == i].last_ig.values)) & (fires[fires.fireID == i].t < str(time_dist[time_dist.fireID == i].first_detect.values))
# tmp_late = (fires[fires.fireID == i].t >= str(*time_dist[time_dist.fireID == i].first_detect.values)) & (fires[fires.fireID == i].t <= str(*tmp.values))

# print(str(*tmp.values))
# time_dist

In [ ]:
early = fires[fires.is_day_after_ig_or_smaller == True].groupby("fireID").FWI.mean().reset_index()
late = fires[fires.is_day_after_fd == True].groupby("fireID").FWI.mean().reset_index()
late = late.rename(columns={"FWI":"FWI_post_ig"})

early = early.merge(late, on = ["fireID"], how = "outer")
early["diff"] = early.FWI - early.FWI_post_ig
early = early.merge(time_dist, on = ('fireID'))
print(early['diff'].mean())
early

In [ ]:
#plt.scatter(early["diff"], early.time_diff.dt.days)

In [ ]:
plt.hist(early['diff'])

In [ ]:
early = fires[fires.is_period_between_ig_fd == True].groupby("fireID").FWI.mean().reset_index()
late = fires[fires.is_period_after_fd == True].groupby("fireID").FWI.mean().reset_index()
late = late.rename(columns={"FWI":"FWI_post_ig"})

early = early.merge(late, on = ["fireID"], how = "outer")
early["diff"] = early.FWI - early.FWI_post_ig
early = early.merge(time_dist, on = ('fireID'))
print(early["diff"].mean())
early

In [ ]:
plt.hist(early['diff'])

In [ ]:
plt.scatter(early["diff"], early.time_diff.dt.days)

In [ ]:
### Check how many "non lightning ignited fires" have a spatial intersection with a "lightning ignited fire"

# lt_true = plot_last_tmp[plot_last_tmp.has_lt == True]
# lt_false = plot_last_tmp[plot_last_tmp.has_lt == False]

# lt_test = lt_false.sjoin(lt_true)
# lt_test[['fireID_left', 't_left', 'geometry', 
         
#        'fireID_right', 't_right',  'has_lt_right']]

In [ ]:
# print("There are ", len(lt_test.fireID_left.unique()), " ignition-less fire perimeters that overlap with a lightning ignition")
# print("out of ", len(lt_false.fireID.unique()),  "total ignition-less fires")

# print("when there are ", len(lt_true.fireID.unique()),  " fires with confirmed strikes")


# ## Check difference bewtween big fires

# lt_test["t_diff"] = lt_test.t_right.astype("datetime64[ns]") - lt_test.t_left.astype("datetime64[ns]")


# lt_test["t_diff"].astype('timedelta64[h]').plot.hist(title = "time between fires that got sorted into different fires")

# print(lt_test["t_diff"].max())
# print(lt_test["t_diff"].min())

# lt_test[lt_test["t_diff"] == lt_test["t_diff"].max()].fireID_left

# print("There are ", len(lt_test[lt_test["t_diff"].astype("int") > 0].fireID_left.unique()), "fires where strikes happened before")


# lt_test[lt_test["t_diff"].astype("int") > 0].t_diff.astype('timedelta64[h]').plot.hist(title = "time between fires that got sorted into different fires")
# lt_test[lt_test["t_diff"].astype("int") > 0].t_diff.astype('timedelta64[h]').mean()/24

In [ ]:
# gpd_fires

In [ ]:
# first_perims = gpd_fires[~gpd_fires.geometry.isnull()].groupby("fireID").t.min()
# first_perims

# last_perims = gpd_fires[~gpd_fires.geometry.isnull()].groupby("fireID").t.max().reset_index()
# last_perims

# plot_last = last_perims.merge(gpd_fires[~gpd_fires.geometry.isnull()], on = ["fireID", "t"], how = "left")

In [ ]:
# no_lt_ids

In [ ]:
# year = '2023'
# fireID = '12146'
# check_last = False
# path_region="BOREAL_NRT_3571_DPS"  
# check_last = False 
# FWI_source = "station"

# ## Read in the largefire file of the fireID
# try:
#     fr = fv.load_large_fire(fireID, year = year, path_region= path_region, s3_path = True) ## Cluster of 2 fires. 
# except Exception as e:
#     print("Fire ID cannot be opened:",fireID)
#     print(e)

#     ## TO DO Filter? 
#         ## VIIRS Static source filter?
#         ## WUI filter? 

# fr = fr.to_crs("3571")
# oldest_perim = fr[fr.t == fr.t.max()]
# first_perim = fr[fr.t == fr.t.min()]

# if(check_last == True):
#     oldest_perim.t = oldest_perim.t.astype("str")
#     m = oldest_perim.explore()





#  ## Subset lightning by time and space

# ## TO DO: Figure out which CA ecoregion/province the fire is in and subset lighting by that? 
# #print("Not yet subseting spatially beyond quebec. Assuming quebec bounding box")

# min_threshold = fr.t.astype('datetime64[ns]').min() - timedelta(days = 10)
# possible_lt = lt[lt.t <= fr.t.min()]
# possible_lt = possible_lt[possible_lt.t >= min_threshold]

# #first_perim.geometry = first_perim.buffer(750*2) ## Two viirs pixels???
# join_lt = gpd.sjoin(possible_lt, first_perim, predicate = 'within', how = "inner")
# join_lt["no_strikes_in_time"] = False

# if (len(possible_lt == 0)):
#     join_lt["no_strikes_in_time"] = True

# if (len(join_lt[join_lt.InterCloud.isna()].InterCloud) == len(join_lt.InterCloud)):
#     join_lt["num_candidates"] = 0
#     join_lt["num_strikes"] = len(possible_lt)
#     join_lt["num_strikes_10_days"] = len(possible_lt)
# else:
#     ## Extract "denominator" or the # of strikes from same period
#     denominator = possible_lt[possible_lt.t >= join_lt.t_left.min()]
#     denominator = denominator[denominator.t <= join_lt.t_left.max()]
#     join_lt["num_candidates"] = len(join_lt)
#     join_lt["num_strikes"] = len(denominator)
#     join_lt["num_strikes_10_days"] = len(possible_lt)

# ## Get distance to individuals ignitions
# # fr["perim_rank"] = fr.t.rank()
# # first_geom = fr[fr.perim_rank == 1].geometry
# # first_geom = first_geom.iloc[0]
# # num_starts = len(first_geom.geoms)
# # for i in range(0, num_starts):
# #     join_lt["dist_start_" + str(i)] = join_lt.distance(first_geom.geoms[i].centroid)
# #     print(fr[fr.perim_rank == 1].to_crs("4326").geometry.iloc[0].geoms[i].centroid)

# ## Rank candidate by distance
# # range_geoms = list(range(0, num_starts))
# # string = "dist_start_"
# # columns_dists = [string + str(x) for x in range_geoms]
# # top = len(join_lt) * 1 # Top 100%. Could cut to smaller range
# # dist_bool = join_lt[columns_dists].rank() <= top ## NEED a max distance cutoff. 
# # join_lt["candidate"] = dist_bool.any(axis = 1)

# ## Get raw VIIRS pixel timing
# date_string = fr.t.astype("datetime64[ns]").max().strftime("%Y%m%d%p")
# print(date_string)
# raw_obs_times = fv.raw_pixel_times(int(fireID), date_string = date_string, path_region = path_region)
# raw_obs_times = raw_obs_times.reset_index()

# ## get station data
# if(FWI_source == "station"):
#     print("Assuming Single Quebec Station. 718270-99999.")
#     st = pd.read_csv("s3://veda-data-store-staging/EIS/other/station-FWI/19900101.NRT/FWI/718270-99999.linear.HourlyFWIFromHourlyInterpContinuous.csv") ## Corrected record from Robert
#     st.HH = st.HH.astype("int")
#     st.YYYY = st.YYYY.astype("int")
#     st.MM = st.MM.astype("int")
#     st.DD = st.DD.astype("int")
#     st = fv.date_convert(st)

#     st_rm = st[["time", "TEMP_C", 'RH_PERC', 'VPD_HPA', 'WDSPD_KPH',
#    'PREC_MM', 'SNOWD_M', 'VIS_KM', 'FFMC', 'DMC', 'DC', 'BUI', 'ISI',
#    'FWI', 'OBSMINUTEDIFF_TEMP', 'OBSMINUTEDIFF_RH', 'OBSMINUTEDIFF_WDSPD',
#    'ISPRECREPORTED', 'OBSMINUTEDIFF_SNOW', 'OBSMINUTEDIFF_VIS']]
#     st_rm = st_rm.rename(columns = {"time":"t"})
#     #### Subset station data by time. 
#     st_rm = st_rm[st_rm.t >= min_threshold]
#     st_rm = st_rm[st_rm.t <= fr.t.max()]

# else:
#     #print("No other FWI extraction method ready. Sorry. ")
#     raise Exception("No other FWI extraction method ready. Sorry. ")

# ## Do merging of all dfs 
# foo = join_lt[["InterCloud", "t_left", "lat_left", "lon_left", "current_mag", "error_elps", "num_station", "num_candidates", "num_strikes", "num_strikes_10_days", "no_strikes_in_time"]]
# foo = foo.rename(columns = {"t_left":"t", "lat_left":"lat", "lon_left":"lon"})
# foo.t = foo.t.astype('datetime64[ns]')
# raw_obs_times = raw_obs_times.rename(columns={"count": "viirs_pix_count"}) 
# raw_obs_times.t = raw_obs_times.t.astype("datetime64[ns]")
# merged = foo.merge(raw_obs_times, on = ["t"], how = "outer")

# fr_rm = fr.rename(columns = {"lat":"lat_centroid", "lon":"lon_centroid"})
# fr_rm.t = fr_rm.t.astype("datetime64[ns]")
# merged = merged.merge(fr_rm, on = ["t"], how = "outer")

# merged = merged.merge(st_rm, on = ["t"], how = "outer")
# merged["fireID"] = fireID

# ## Find temporal thresholds

# first_ig = merged[~merged.InterCloud.isna()].t.min()
# last_ig = merged[~merged.InterCloud.isna()].t.max()
# first_detection = merged[~merged.viirs_pix_count.isna()].t.min()


# merged["pre_fire"] = ((merged.t >= last_ig) & (merged.t <=  first_detection)) #### first_ig better????????????

In [ ]:
# #possible_lt
# join_lt

In [ ]:
# first_perim.t = first_perim.t.astype("str")
# first_perim.explore()

In [ ]:
# first_perim.buffer(750*2).explore() ## Two viirs pixels???

# #first_perim.crs